In [1]:
import pandas as pd
import numpy as np

# Load dữ liệu thông minh đã tổng hợp từ 5 buổi trước
df = pd.read_csv(r'C:\Users\Hi\Desktop\RetentIO_Project\data\final_bank_customer_intelligence.csv')

# Convert lại cột id thành string để tra cứu cho dễ
df['id'] = df['id'].astype(str)

print(f"RetentIO Agent đã sẵn sàng! Đang quản lý {df.shape[0]} khách hàng.")
print(df[['id', 'full_name', 'CLV_Segment', 'Uplift_Segment']].head())

RetentIO Agent đã sẵn sàng! Đang quản lý 79949 khách hàng.
  id         full_name CLV_Segment Uplift_Segment
0  1       Đặng Văn Vũ   Mid Value    Sure Things
1  2      Bùi Hữu Phúc   Low Value   Persuadables
2  3        Lê Văn Mai   Low Value    Sure Things
3  4  Dương Trần Nhiên  High Value   Persuadables
4  5    Dương Thu Linh   Low Value   Persuadables


In [2]:
def define_strategy(profile):
    """
    Hàm này đóng vai trò là 'Chiến lược gia' (The Strategist).
    Input: Một dòng dữ liệu khách hàng (Series).
    Output: Hành động cụ thể (Action) & Gói ưu đãi (Offer).
    """
    
    # 1. Nhóm CẤM LÀM PHIỀN (Sleeping Dogs)
    if profile['Uplift_Segment'] == 'Sleeping Dogs':
        return {
            'action': 'DO_NOT_DISTURB',
            'priority': 'Lowest',
            'offer': 'None',
            'channel': 'None'
        }

    # 2. Nhóm CẦN GIỮ CHÂN (Persuadables)
    if profile['Uplift_Segment'] == 'Persuadables':
        # Nếu là VIP (High Value) -> Chăm sóc đặc biệt
        if profile['CLV_Segment'] == 'High Value':
            return {
                'action': 'RETENTION_VIP',
                'priority': 'Critical (High)',
                'offer': 'Tặng 500k tiền mặt + Miễn phí thường niên trọn đời',
                'channel': 'Direct Call (Relationship Manager)'
            }
        # Nếu là khách thường -> Gửi Voucher tự động
        else:
            return {
                'action': 'RETENTION_MASS',
                'priority': 'High',
                'offer': 'Voucher hoàn tiền 50k cho giao dịch tiếp theo',
                'channel': 'Zalo/SMS Automation'
            }

    # 3. Nhóm CHẮC CHẮN Ở LẠI (Sure Things) -> Tranh thủ bán thêm (Cross-sell)
    if profile['Uplift_Segment'] == 'Sure Things':
        # Gợi ý sản phẩm dựa trên số dư
        product = "Thẻ Tín dụng Platinum" if profile['balance'] > 50_000_000 else "Gửi tiết kiệm Online lãi suất cao"
        return {
            'action': 'CROSS_SELL',
            'priority': 'Medium',
            'offer': f'Mời mở {product}',
            'channel': 'Email Marketing'
        }

    # 4. Nhóm VÔ PHƯƠNG (Lost Causes)
    return {
        'action': 'IGNORE',
        'priority': 'Low',
        'offer': 'None',
        'channel': 'None'
    }

print("Đã nạp xong chiến thuật kinh doanh!")

Đã nạp xong chiến thuật kinh doanh!


In [3]:
def generate_prompt_for_llm(profile, strategy):
    """
    Tạo prompt để gửi cho LLM (ChatGPT/Gemini) viết nội dung.
    """
    if strategy['action'] in ['DO_NOT_DISTURB', 'IGNORE']:
        return None

    # Context cho LLM
    context = f"""
    Bạn là Trợ lý AI của Ngân hàng Techcombank. Hãy viết một nội dung ngắn gọn, thân thiện gửi tới khách hàng.
    
    THÔNG TIN KHÁCH HÀNG:
    - Tên: {profile['full_name']}
    - Giới tính: {profile['gender']}
    - Nghề nghiệp: {profile['occupation']}
    - Số dư hiện tại: {profile['balance']:,.0f} VNĐ
    - Phân khúc: {profile['CLV_Segment']}
    
    MỤC TIÊU CHIẾN DỊCH:
    - Hành động: {strategy['action']}
    - Ưu đãi (Offer): {strategy['offer']}
    
    YÊU CẦU:
    - Giọng văn: Chuyên nghiệp nhưng gần gũi.
    - Nhấn mạnh vào lợi ích của Offer.
    - Nếu là VIP, hãy tỏ ra trân trọng.
    - Độ dài: Dưới 100 từ.
    """
    return context

In [4]:
import random

# Hàm giả lập LLM (Nếu bạn có API Key, thay hàm này bằng call API thực tế)
def mock_llm_response(strategy, name):
    if strategy['action'] == 'RETENTION_VIP':
        return f"[DRAFT EMAIL] Kính gửi Quý khách {name}, Techcombank trân trọng sự gắn bó của Quý khách. Để tri ân, chúng tôi xin gửi tặng đặc quyền Miễn phí thường niên..."
    elif strategy['action'] == 'RETENTION_MASS':
        return f"[SMS] Chao ban {name}, nho ban qua! Tang ban ngay 50k cho giao dich tiep theo tai Techcombank. Dung bo lo nhe!"
    elif strategy['action'] == 'CROSS_SELL':
        return f"[APP NOTI] {name} ơi, tiền nhàn rỗi để không thì phí quá! Mở tiết kiệm Online ngay lãi suất tới 6% nhé."
    return ""

def run_agent(customer_id):
    # 1. Fetch Data
    try:
        profile = df[df['id'] == str(customer_id)].iloc[0]
    except IndexError:
        return "❌ Error: Customer ID not found."

    # 2. Decide Strategy
    strategy = define_strategy(profile)

    # 3. Generate Content
    prompt = generate_prompt_for_llm(profile, strategy)
    
    # Giả lập kết quả từ LLM
    content = mock_llm_response(strategy, profile['full_name']) if prompt else "(No Content Generated)"

    # 4. Display Report
    print(f"\n{'='*40}")
    print(f"🤖 RETENTIO AGENT REPORT | ID: {customer_id}")
    print(f"{'='*40}")
    print(f"👤 Khách hàng: {profile['full_name']} ({profile['occupation']})")
    print(f"📊 Chỉ số: Risk={profile['Survival_Risk_Score']:.2f} | Uplift={profile['Uplift_Segment']}")
    print(f"💰 Phân khúc: {profile['CLV_Segment']} (Bal: {profile['balance']:,.0f})")
    print(f"{'-'*40}")
    print(f"🎯 QUYẾT ĐỊNH CHIẾN LƯỢC: {strategy['action']}")
    print(f"⚡ Độ ưu tiên: {strategy['priority']}")
    print(f"🎁 Offer: {strategy['offer']}")
    print(f"📡 Kênh: {strategy['channel']}")
    print(f"{'-'*40}")
    if prompt:
        print(f"📝 NỘI DUNG ĐỀ XUẤT (AI Generated):\n{content}")
    else:
        print("🤫 ACTION: Giữ im lặng (Theo chiến thuật)")
    print(f"{'='*40}\n")

In [5]:
# Chọn 3 khách hàng đại diện cho 3 tình huống khác nhau
# 1. Một người VIP cần giữ (Persuadable + High Value)
vip_user = df[(df['Uplift_Segment'] == 'Persuadables') & (df['CLV_Segment'] == 'High Value')].iloc[0]['id']

# 2. Một người cần tránh (Sleeping Dog)
sleep_user = df[df['Uplift_Segment'] == 'Sleeping Dogs'].iloc[0]['id']

# 3. Một người an toàn để bán thêm (Sure Thing)
sure_user = df[df['Uplift_Segment'] == 'Sure Things'].iloc[0]['id']

# Chạy Demo
print(">>> TÌNH HUỐNG 1: KHÁCH VIP ĐANG CHÁN")
run_agent(vip_user)

print(">>> TÌNH HUỐNG 2: KHÁCH HÀNG NHẠY CẢM")
run_agent(sleep_user)

print(">>> TÌNH HUỐNG 3: KHÁCH HÀNG TRUNG THÀNH")
run_agent(sure_user)

>>> TÌNH HUỐNG 1: KHÁCH VIP ĐANG CHÁN

🤖 RETENTIO AGENT REPORT | ID: 4
👤 Khách hàng: Dương Trần Nhiên (Chủ Doanh nghiệp nhỏ)
📊 Chỉ số: Risk=0.51 | Uplift=Persuadables
💰 Phân khúc: High Value (Bal: 50,615,501)
----------------------------------------
🎯 QUYẾT ĐỊNH CHIẾN LƯỢC: RETENTION_VIP
⚡ Độ ưu tiên: Critical (High)
🎁 Offer: Tặng 500k tiền mặt + Miễn phí thường niên trọn đời
📡 Kênh: Direct Call (Relationship Manager)
----------------------------------------
📝 NỘI DUNG ĐỀ XUẤT (AI Generated):
[DRAFT EMAIL] Kính gửi Quý khách Dương Trần Nhiên, Techcombank trân trọng sự gắn bó của Quý khách. Để tri ân, chúng tôi xin gửi tặng đặc quyền Miễn phí thường niên...

>>> TÌNH HUỐNG 2: KHÁCH HÀNG NHẠY CẢM

🤖 RETENTIO AGENT REPORT | ID: 29
👤 Khách hàng: Phạm Gia Thảo (Chủ Doanh nghiệp nhỏ)
📊 Chỉ số: Risk=0.08 | Uplift=Sleeping Dogs
💰 Phân khúc: Low Value (Bal: 217,862,308)
----------------------------------------
🎯 QUYẾT ĐỊNH CHIẾN LƯỢC: DO_NOT_DISTURB
⚡ Độ ưu tiên: Lowest
🎁 Offer: None
📡 Kênh: No